# Spectral probe (D244, D245)

Do the stored representations hold the spectrum of their input, readable by a linear probe, and do the
methods that keep more of it classify better?

* `results/<dataset>.csv`: test R² of a ridge probe per spectral target (`probe_spectral.py`, D244)
* `results/<dataset>_classify.csv`: premise test (a), the spectral targets themselves as features for
  the benchmark's logreg probe (`classify_spectral.py`, D245)
* `results/<dataset>_bands.csv`: premise test (c), each relative band energy on its own as the
  feature of the benchmark's logreg probe, all training windows (`classify_spectral.py`, D252)
* premise test (b), below: each run's spectral R² against its benchmark logreg accuracy
  (`runs/<run>/results.csv`, seed 0)

Run from the repo root folder (paths are relative to it). Clear the outputs before committing.

In [ ]:
import glob
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

if os.path.basename(os.getcwd()) == "spectral_probe":  # opened from its own folder: paths are relative to the repo root
    os.chdir(os.path.join("..", ".."))
RESULTS = "studies/spectral_probe/results"
DATASETS = ["forda", "cwru", "hust", "paderborn", "xjtu_sy"]
ACC_FRACTION = 1.0  # label fraction of the class accuracies in (a) and (b): 0.01, 0.1 or 1.0 (CWRU: 0.1, 1.0)

probe = {d: pd.read_csv(f"{RESULTS}/{d}.csv") for d in DATASETS if os.path.exists(f"{RESULTS}/{d}.csv")}
classify = {d: pd.read_csv(f"{RESULTS}/{d}_classify.csv") for d in DATASETS if os.path.exists(f"{RESULTS}/{d}_classify.csv")}
bands = {d: pd.read_csv(f"{RESULTS}/{d}_bands.csv") for d in DATASETS if os.path.exists(f"{RESULTS}/{d}_bands.csv")}
print("probe results:", list(probe), "| premise test (a):", list(classify), "| premise test (c):", list(bands))

## Mean R² over the bands

Mean test R² over the 16 relative band energies (log share of the total; the target every dataset has),
per run, probes trained on all training windows (D246). Paderborn: mean over its 10 combinations. `floor_mean` = predicting the
training mean (negative when the test windows' mean differs from the training mean, e.g. other bearings).

In [ ]:
def band_mean(d, target="relative"):
    """Mean R² over the bands of one target kind, per name, probe and label fraction (mean over channels and
    Paderborn's combinations)."""
    r = d[d.target == target]
    return r.groupby(["name", "probe", "label_fraction"]).r2.mean().unstack()

for name, d in probe.items():
    print(f"=== {name}")
    t = band_mean(d)
    display(t.xs("ridge", level="probe").round(3))
    print("floor_mean:", t.xs("floor_mean", level="probe").mean().round(2).to_dict())

## R² per band

Ridge R² for each relative band energy (all training windows, D246), against the band's centre frequency (fraction of the
sample rate). FFT reference: black, dashed. Random-init encoders (`_random`, `_random_bn`): grey.

In [ ]:
def style(name):
    if name == "fft_reference":
        return dict(color="black", ls="--", lw=2, zorder=3)
    if "_random" in name:
        return dict(color="grey", lw=0.8, alpha=0.6)
    return dict(lw=1.2)

for dname, d in probe.items():
    r = d[(d.probe == "ridge") & (d.target == "relative") & (d.label_fraction == 1.0)]
    channels = r.channel.unique()
    fig, axes = plt.subplots(1, len(channels), figsize=(7 * len(channels), 4.5), squeeze=False)
    for ax, ch in zip(axes[0], channels):
        c = r[r.channel == ch]
        for name, g in c.groupby("name"):
            g = g.groupby("band")[["f_lo", "f_hi", "r2"]].mean()  # Paderborn: mean over combinations
            ax.plot((g.f_lo + g.f_hi) / 2, g.r2, label=name, **style(name))
        ax.set(title=f"{dname}: {ch}", xlabel="band centre (cycles per sample; Nyquist = 0.5)", ylabel="test R²",
               ylim=(max(-1, ax.get_ylim()[0]), 1.02))
        ax.axhline(0, color="k", lw=0.5)
    axes[0][-1].legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

## Premise test (a): the spectral targets as class features

Benchmark logreg probe on the spectral targets alone (FordA, XJTU-SY: 16 relative energies + centroid
per channel; CWRU, HUST, Paderborn: 16 absolute + 16 relative + total + centroid). Paderborn: mean and
SD over the 10 combinations, at each of the runs' label fractions.

In [ ]:
for name, c in classify.items():
    print(f"=== {name}")
    display(c.groupby(["probe", "label_fraction"]).accuracy.agg(["mean", "std", "count"]).round(3))

## Premise test (c): how well does each band classify on its own?

The benchmark logreg probe on a single feature, one band's relative energy, with all training windows
(D252); test accuracy against the band's centre frequency. Grey dashed: the floor (most frequent class).
Paderborn: mean over its 10 combinations.

In [ ]:
for dname, c in bands.items():
    channels = c.channel.unique()
    fig, axes = plt.subplots(1, len(channels), figsize=(7 * len(channels), 3.5), squeeze=False)
    for ax, ch in zip(axes[0], channels):
        g = c[c.channel == ch].groupby(["probe", "band"])[["f_lo", "f_hi", "accuracy"]].mean()
        lr, fl = g.loc["logreg"], g.loc["floor"]
        ax.plot((lr.f_lo + lr.f_hi) / 2, lr.accuracy, marker="o", label="logreg, one band")
        ax.plot((fl.f_lo + fl.f_hi) / 2, fl.accuracy, color="grey", ls="--", label="floor")
        ax.set(title=f"{dname}: {ch}", xlabel="band centre (cycles per sample; Nyquist = 0.5)", ylabel="test accuracy")
    axes[0][-1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## Premise test (b): spectral R² against classification accuracy

One point per run (Paderborn: mean over combinations): x = mean ridge R² over the relative band energies,
trained on all training windows (D246); y = the run's benchmark logreg test accuracy (seed 0) at `ACC_FRACTION`.
Dashed line: premise test (a) at the same fraction. Spearman's ρ over the trained SSL runs (not the references, baselines or random-init
encoders). XJTU-SY: the health-stage runs only (the RUL runs have no class accuracy).

In [ ]:
BASELINES = ("catch22", "minirocket")  # FordA reference rows with their own benchmark accuracy

def benchmark_accuracy(run, fraction):
    """The run's logreg test accuracy at the fraction (seed 0), or None (no results.csv, or a regression run)."""
    path = os.path.join("runs", run, "results.csv")
    if not os.path.exists(path):
        print(f"no results.csv: {run}")
        return None
    res = pd.read_csv(path)
    if "accuracy" not in res:
        return None
    row = res[(res.probe == "logreg") & (res.label_fraction == fraction) & (res.seed == 0)]
    return float(row.accuracy.iloc[0]) if len(row) else None

for dname, d in probe.items():
    r = d[(d.probe == "ridge") & (d.target == "relative") & (d.label_fraction == 1.0) & (d.name != "fft_reference")]
    per_run = r.groupby(["run", "name"]).r2.mean().reset_index()
    per_run["accuracy"] = [benchmark_accuracy(run, ACC_FRACTION) for run in per_run.run]
    pts = per_run.dropna(subset=["accuracy"]).groupby("name")[["r2", "accuracy"]].mean()  # Paderborn: over combinations
    if pts.empty:
        continue
    kind = np.where(pts.index.str.contains("_random"), "random-init",
                    np.where(pts.index.str.startswith(BASELINES), "baseline", "SSL"))
    fig, ax = plt.subplots(figsize=(7, 5))
    for k, marker, colour in (("SSL", "o", "C0"), ("random-init", "x", "grey"), ("baseline", "s", "C1")):
        p = pts[kind == k]
        ax.scatter(p.r2, p.accuracy, marker=marker, color=colour, label=k)
        for n, row in p.iterrows():
            ax.annotate(n, (row.r2, row.accuracy), fontsize=6, xytext=(3, 2), textcoords="offset points")
    if dname in classify:
        a = classify[dname]
        a = a[(a.probe == "logreg") & (a.label_fraction == ACC_FRACTION)].accuracy.mean()
        ax.axhline(a, color="black", ls="--", lw=1, label="(a) spectral targets")
    ssl = pts[kind == "SSL"]
    rho = spearmanr(ssl.r2, ssl.accuracy).statistic if len(ssl) > 2 else np.nan
    ax.set(title=f"{dname}, accuracy at {ACC_FRACTION:.0%} of the labels: Spearman ρ (SSL runs) = {rho:.2f}, n = {len(ssl)}",
           xlabel="mean R², relative band energies (all training windows)", ylabel="benchmark logreg accuracy")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()